In [ ]:
%pip install openpyxl
dbutils.library.restartPython()

In [ ]:
spark.sql("DELETE FROM asset_api_webapp_dev.default.kpi_state")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from typing import Dict, Any, List
import uuid
import json
from datetime import datetime
import os
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd

# =========================
# CONFIG
# =========================
s3_incoming   = "s3://demo-document-bucket/incoming/"
s3_accepted   = "s3://demo-document-bucket/accepted/"
s3_rejected   = "s3://demo-document-bucket/rejected/"
s3_logs_rej   = "s3://demo-document-bucket/logs/rejected/"  # 1 log par fichier rejeté
s3_logs_api = "s3://demo-document-bucket/logs/api_upload/"
checkpoint_path = "dbfs:/tmp/asset_api_kpi_ingestion/_processed_files"
checkpoint_table = "asset_api_webapp_dev.default.kpi_state"

# =========================
# CONFIG VOLUME (Unity Catalog)
# =========================
VOLUME_EXCEL_DIR_DBFS  = "dbfs:/Volumes/asset_api_webapp_dev/default/kpi_parquet/excel_exports/"
# (équivalent local pour open())
# /dbfs/Volumes/asset_api_webapp_dev/default/kpi_parquet/excel_exports/

VOLUME_EXCEL_DIR = "/Volumes/asset_api_webapp_dev/default/kpi_parquet/excel_exports/"


N_MAX = 1000
SAMPLE_N = 50
run_ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

# =========================
# HELPERS
# =========================
def safe_ls(path: str):
    try:
        return dbutils.fs.ls(path)
    except Exception:
        return []

def ensure_dir(path: str):
    try:
        dbutils.fs.mkdirs(path)
    except Exception:
        pass

def move_to(src_path: str, dest_dir: str) -> str:
    ensure_dir(dest_dir)
    name = src_path.split("/")[-1]
    dest = dest_dir.rstrip("/") + "/" + name
    dbutils.fs.mv(src_path, dest)
    return dest

def write_text(path: str, content: str):
    ensure_dir(path.rsplit("/", 1)[0] + "/")
    dbutils.fs.put(path, content, overwrite=True)

def normalize_kpi_stg(kpi: DataFrame) -> DataFrame:
    return (kpi
        .withColumn(
            "Period",
            F.when(
                F.col("PERIOD").rlike(r"^\d{2}/\d{4}$"),
                F.concat_ws(".", F.substring("PERIOD", 4, 4), F.substring("PERIOD", 1, 2))
            ).otherwise(F.col("PERIOD"))
        )
        .withColumnRenamed("DOMAIN", "data_domain_code")
        .withColumnRenamed("UNIT", "asset_code")
        .withColumnRenamed("VERSION", "version_code")
        .withColumnRenamed("SCENARIO", "scenario_code")
        .withColumnRenamed("ITEM", "item_code")
    )

def validate_kpi_parquet_simple(
    kpi_stg: DataFrame,
    unit_df: DataFrame,
    plant_df: DataFrame,
    country_df: DataFrame,
    data_domain_df: DataFrame,
    version_df: DataFrame,
    scenario_df: DataFrame,
    exercise_df: DataFrame,
    sample_n: int = 20
) -> Dict[str, Any]:

    reasons: List[Dict[str, Any]] = []

    # R2: asset_code absent de unit
    missing_unit = kpi_stg.join(unit_df.select("asset_code").distinct(), on="asset_code", how="left_anti")
    if missing_unit.take(1):
        reasons.append({
            "rule": "UNKNOWN_ASSET_CODE",
            "message": "Le fichier référence au moins un Asset Code absent de la table unit.",
            "examples": missing_unit.select("asset_code").distinct().limit(sample_n).toPandas().to_dict("records")
        })

    # Résolution country_code via asset_code -> unit -> plant -> country
    kpi_country = (kpi_stg
        .join(unit_df, on="asset_code", how="left")
        .join(plant_df, on="plant_id", how="left")
        .join(country_df.select("country_id","country_code"), on="country_id", how="left")
    )

    # R3: country absent (non résolu via refs)
    missing_country = kpi_country.filter(F.col("country_code").isNull())
    if missing_country.take(1):
        reasons.append({
            "rule": "UNKNOWN_COUNTRY",
            "message": "Le fichier référence un country inexistant (via asset_code -> unit -> plant -> country).",
            "examples": (missing_country
                         .select("asset_code","plant_id","country_id")
                         .distinct()
                         .limit(sample_n)
                         .toPandas()
                         .to_dict("records"))
        })

    # R4: data_domain absent
    missing_dd = kpi_stg.join(data_domain_df.select("data_domain_code").distinct(),
                              on="data_domain_code", how="left_anti")
    if missing_dd.take(1):
        reasons.append({
            "rule": "UNKNOWN_DATA_DOMAIN",
            "message": "Le fichier référence un Data Domain absent de la table data_domain.",
            "examples": missing_dd.select("data_domain_code").distinct().limit(sample_n).toPandas().to_dict("records")
        })

    # R1: doublons (dans le fichier)
    dup_keys = ["data_domain_code", "country_code", "version_code", "scenario_code", "Period", "item_code", "asset_code"]
    dups = (kpi_country
        .groupBy(*dup_keys)
        .count()
        .filter(F.col("count") > 1)
    )
    if dups.take(1):
        reasons.append({
            "rule": "DUPLICATE_KEYS",
            "message": f"Doublons détectés: >1 enregistrement pour la même clé {dup_keys} (même si QUANTITY diffère).",
            "examples": dups.limit(sample_n).toPandas().to_dict("records")
        })

    # R5: couples (version_code, scenario_code) absents de exercise
    kpi_vs = (kpi_stg
        .join(version_df, on="version_code", how="left")
        .join(scenario_df, on="scenario_code", how="left")
    )

    unknown_version = kpi_vs.filter(F.col("version_id").isNull()).select("version_code").distinct()
    if unknown_version.take(1):
        reasons.append({
            "rule": "UNKNOWN_VERSION_CODE",
            "message": "Le fichier référence au moins un Version code absent de la table version.",
            "examples": unknown_version.limit(sample_n).toPandas().to_dict("records")
        })

    unknown_scenario = kpi_vs.filter(F.col("scenario_id").isNull()).select("scenario_code").distinct()
    if unknown_scenario.take(1):
        reasons.append({
            "rule": "UNKNOWN_SCENARIO_CODE",
            "message": "Le fichier référence au moins un Scenario code absent de la table scenario.",
            "examples": unknown_scenario.limit(sample_n).toPandas().to_dict("records")
        })

    pairs = (kpi_vs
        .select("version_id", "scenario_id")
        .distinct()
        .filter(F.col("version_id").isNotNull() & F.col("scenario_id").isNotNull())
    )

    missing_pairs = pairs.join(exercise_df.distinct(), on=["version_id","scenario_id"], how="left_anti")
    if missing_pairs.take(1):
        missing_pairs_codes = (missing_pairs
            .join(version_df, on="version_id", how="left")
            .join(scenario_df, on="scenario_id", how="left")
            .select("version_code","scenario_code")
            .distinct()
        )
        reasons.append({
            "rule": "UNKNOWN_VERSION_SCENARIO_IN_EXERCISE",
            "message": "Au moins un couple (Version, Scenario) du fichier n'existe pas dans exercise.",
            "examples": missing_pairs_codes.limit(sample_n).toPandas().to_dict("records")
        })

    return {"is_valid": len(reasons) == 0, "reasons": reasons}


def dbfs_to_local(dbfs_path: str) -> str:
    # "dbfs:/a/b" -> "/dbfs/a/b" (utilisable avec open())
    if not dbfs_path.startswith("dbfs:/"):
        raise ValueError(f"Not a dbfs:/ path: {dbfs_path}")
    return "/dbfs/" + dbfs_path[len("dbfs:/"):]

def export_kpi2_to_excel_in_volume(kpi2_df: DataFrame, out_dbfs_path: str) -> str:
    """
    Écrit un Excel dans le Volume (via /dbfs/Volumes/...).
    Retourne le chemin local pour upload (open()).
    """
    out_local = dbfs_to_local(out_dbfs_path)
    os.makedirs(os.path.dirname(out_local), exist_ok=True)

    pdf = kpi2_df.toPandas()
    with pd.ExcelWriter(out_local, engine="openpyxl") as writer:
        pdf.to_excel(writer, index=False, sheet_name="kpi2")

    return out_local

def export_kpi2_to_excel_in_posix_volume(kpi2_df: DataFrame, out_posix_path: str) -> str:
    """
    Écrit directement un .xlsx dans /Volumes/... (POSIX).
    Retourne le même chemin (utilisable par open()).
    ⚠️ Utilise toPandas() -> OK si volume raisonnable.
    """
    os.makedirs(os.path.dirname(out_posix_path), exist_ok=True)

    pdf = kpi2_df.toPandas()
    with pd.ExcelWriter(out_posix_path, engine="openpyxl") as writer:
        pdf.to_excel(writer, index=False, sheet_name="kpi2")

    return out_posix_path

def export_kpi2_to_excel_local_only(kpi2_df: DataFrame) -> str:
    """
    Ecrit l'Excel en local (/tmp) uniquement (serverless-safe).
    Retourne le chemin local pour upload.
    """
    local_dir = f"/tmp/asset_api_excel/{run_ts}"
    os.makedirs(local_dir, exist_ok=True)
    local_xlsx = f"{local_dir}/{uuid.uuid4().hex}.xlsx"

    pdf = kpi2_df.toPandas()
    with pd.ExcelWriter(local_xlsx, engine="openpyxl") as writer:
        pdf.to_excel(writer, index=False, sheet_name="kpi2")

    return local_xlsx

def export_kpi2_excel_local_then_copy_to_volume(kpi2_df: DataFrame, out_volume_dbfs_path: str) -> str:
    """
    Serverless-safe:
    1) écrit l'Excel en local driver (/tmp)
    2) copie vers le Volume via dbutils.fs.cp(file:..., dbfs:/Volumes/...)
    Retourne le chemin local (/tmp/...) à utiliser pour l'upload API.
    """
    if not out_volume_dbfs_path.startswith("dbfs:/Volumes/"):
        raise ValueError(f"Expected dbfs:/Volumes/... but got: {out_volume_dbfs_path}")

    # 1) write local
    local_dir = f"/tmp/asset_api_excel/{run_ts}"
    os.makedirs(local_dir, exist_ok=True)
    local_xlsx = f"{local_dir}/{uuid.uuid4().hex}.xlsx"

    pdf = kpi2_df.toPandas()
    with pd.ExcelWriter(local_xlsx, engine="openpyxl") as writer:
        pdf.to_excel(writer, index=False, sheet_name="kpi2")

    # 2) copy to Volume
    dbutils.fs.mkdirs(out_volume_dbfs_path.rsplit("/", 1)[0])
    # overwrite compatible serverless: rm puis cp
    try:
        dbutils.fs.rm(out_volume_dbfs_path, True)
    except Exception:
        pass

    dbutils.fs.cp(f"file:{local_xlsx}", out_volume_dbfs_path)

    return local_xlsx

# =========================
# 0) Load reference tables once
# =========================
unit_ref = spark.table("asset_api_webapp_dev.default.unit").select(
    F.col("code").alias("asset_code"),
    "plant_id",
    F.col("description").alias("asset_name"),
    "global_technology_id"
)

plant_ref = spark.table("asset_api_webapp_dev.default.plant").select(
    F.col("id").alias("plant_id"),
    "country_id"
)

country_ref = spark.table("asset_api_webapp_dev.default.country").select(
    F.col("id").alias("country_id"),
    F.col("code").alias("country_code"),
    F.col("description").alias("country_desc")
)

data_domain_ref = spark.table("asset_api_webapp_dev.default.data_domain").select(
    F.col("code").alias("data_domain_code")
)

version_ref = spark.table("asset_api_webapp_dev.default.version").select(
    F.col("id").alias("version_id"),
    F.col("code").alias("version_code")
)

scenario_ref = spark.table("asset_api_webapp_dev.default.scenario").select(
    F.col("id").alias("scenario_id"),
    F.col("code").alias("scenario_code")
)

exercise_ref = spark.table("asset_api_webapp_dev.default.exercise").select("scenario_id", "version_id")

gti_ref = spark.table("asset_api_webapp_dev.default.global_technology").select(
    F.col("id").alias("global_technology_id"),
    F.col("main_technology").alias("Technology"),
)

# =========================
# 1) Identify new files (not processed)
# =========================
ensure_dir(s3_accepted); ensure_dir(s3_rejected); ensure_dir(s3_logs_rej); ensure_dir(s3_logs_api)

"""try:
    processed_df = spark.read.format("delta").load(checkpoint_path)
    processed = set(r["path"] for r in processed_df.select("path").collect())
except Exception:
    processed = set()  """ 

try:
    processed_df = spark.table(checkpoint_table)
    processed = set(r["path"] for r in processed_df.select("path").collect())
except Exception:
    processed = set()


files = [f for f in safe_ls(s3_incoming) if f.path.lower().endswith(".parquet")]
new_files = [f for f in files if f.path not in processed]
new_files = sorted(new_files, key=lambda x: x.modificationTime)[:N_MAX]
new_paths = [f.path for f in new_files]

print(f"Nouveaux fichiers trouvés: {len(new_paths)}")
for p in new_paths[:20]:
    print(" -", p)

if len(new_paths) == 0:
    print("Rien à faire.")
    dbutils.notebook.exit("NO_NEW_FILES")


# =========================
# 2) Validate each file individually + move + log
# =========================
accepted_dest_paths = []
rejected_dest_paths = []
checkpoint_rows = []

for src_path in new_paths:
    print(f"\n=== Validation fichier: {src_path} ===")
    try:
        kpi_file = spark.read.parquet(src_path)
        kpi_stg_file = normalize_kpi_stg(kpi_file)

        res = validate_kpi_parquet_simple(
            kpi_stg=kpi_stg_file,
            unit_df=unit_ref.select("asset_code","plant_id"),   # validation only needs these cols
            plant_df=plant_ref,
            country_df=country_ref,
            data_domain_df=data_domain_ref,
            version_df=version_ref,
            scenario_df=scenario_ref,
            exercise_df=exercise_ref,
            sample_n=SAMPLE_N
        )

        if res["is_valid"]:
            dst = move_to(src_path, s3_accepted)
            accepted_dest_paths.append(dst)
            checkpoint_rows.append((src_path, "accepted", run_ts, None))
            print(f"✅ ACCEPTED -> {dst}")

        else:
            dst = move_to(src_path, s3_rejected)
            rejected_dest_paths.append(dst)

            # log par fichier rejeté
            log_payload = {
                "run_ts": run_ts,
                "source_file": src_path,
                "rejected_file": dst,
                "validation": res,
                "notes": {
                    "sample_n": SAMPLE_N
                }
            }
            log_path = f"{s3_logs_rej.rstrip('/')}/rejected_{run_ts}_{src_path.split('/')[-1].replace('.parquet','')}.json"
            write_text(log_path, json.dumps(log_payload, indent=2, ensure_ascii=False))

            checkpoint_rows.append((src_path, "rejected", run_ts, log_path))
            print(f"❌ REJECTED -> {dst}")
            print(f"   Log -> {log_path}")

    except Exception as e:
        # erreur technique de lecture/processing -> rejet + log technique
        dst = move_to(src_path, s3_rejected)
        rejected_dest_paths.append(dst)

        log_payload = {
            "run_ts": run_ts,
            "source_file": src_path,
            "rejected_file": dst,
            "error": str(e)
        }
        log_path = f"{s3_logs_rej.rstrip('/')}/rejected_{run_ts}_{src_path.split('/')[-1].replace('.parquet','')}_TECH.json"
        write_text(log_path, json.dumps(log_payload, indent=2, ensure_ascii=False))

        checkpoint_rows.append((src_path, "rejected_tech", run_ts, log_path))
        print(f"💥 REJECTED (TECH) -> {dst}")
        print(f"   Log -> {log_path}")

# checkpoint update
cp_df = spark.createDataFrame(checkpoint_rows, ["path", "status", "run_ts", "log_path"]) \
    .withColumn("processed_at", F.current_timestamp())

"""(cp_df.write
    .format("delta")
    .mode("append")
    .save(checkpoint_table)
)"""


(cp_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(checkpoint_table)
)


print("\n=== Résumé run ===")
print("Accepted:", len(accepted_dest_paths))
print("Rejected:", len(rejected_dest_paths))


# =========================
# 3) API fichier par fichier (à partir des ACCEPTED)
# =========================
if len(accepted_dest_paths) == 0:
    print("Aucun fichier accepté -> fin du job.")
    dbutils.notebook.exit("NO_ACCEPTED_FILES")


def build_kpi2_for_one_file(kpi_file_df: DataFrame) -> DataFrame:
    kpi = kpi_file_df.withColumn(
        "Period",
        F.when(
            F.col("PERIOD").rlike(r"^\d{2}/\d{4}$"),
            F.concat_ws(".", F.substring("PERIOD", 4, 4), F.substring("PERIOD", 1, 2))
        ).otherwise(F.col("PERIOD"))
    )

    kpi = (kpi
        .withColumnRenamed("DOMAIN", "Data Domain")
        .withColumnRenamed("UNIT", "Asset Code")
        .withColumnRenamed("VERSION", "Version")
        .withColumnRenamed("SCENARIO", "Scenario")
    )

    unit_dim = (unit_ref
        .select(
            F.col("asset_code").alias("Asset Code"),
            F.col("asset_name").alias("Asset Name"),
            "plant_id",
            "global_technology_id"
        )
    )

    plant_dim = plant_ref

    country_dim = country_ref.select(
        "country_id",
        F.col("country_desc").alias("Country")
    )

    kpi2 = (kpi
        .join(unit_dim, on="Asset Code", how="left")
        .join(plant_dim, on="plant_id", how="left")
        .join(country_dim, on="country_id", how="left")
        .withColumn("Geozone", F.col("Country"))
        .join(gti_ref, on="global_technology_id", how="left")
        .select(
            "USERNAME",
            "GID",
            "Data Domain",
            "Geozone",
            "Country",
            "Technology",
            "Asset Code",
            "Asset Name",
            "Version",
            "Scenario",
            "Period",
            "ITEM",
            "QUANTITY",
            "COMMENT",
            "isDeleted"
        )
    )
    return kpi2

# =========================
# CONFIG API + OKTA
# =========================
BASE_URL = "https://service.example.invalid"
UPLOAD_URL = f"{BASE_URL}/kpi-data/upload-excel"
OKTA_TOKEN_URL = "https://service.example.invalid"

CLIENT_ID = dbutils.secrets.get("okta-secret", "client_id_dev")
CLIENT_SECRET = dbutils.secrets.get("okta-secret", "client_secret_dev")


# =========================
# 3) Token + Upload
# =========================
def get_access_token(scope: str = "api.write") -> str:
    r = requests.post(
        OKTA_TOKEN_URL,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={"grant_type": "client_credentials", "scope": scope},
        auth=HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET),
        timeout=30
    )
    if r.status_code != 200:
        raise RuntimeError(f"Okta token error {r.status_code}: {r.text}")
    return r.json()["access_token"]

def upload_excel(excel_file_path: str, data_domain_id: str, gid: str) -> dict:
    token = get_access_token("api.write")  # ✅ token régénéré pour chaque fichier

    headers = {
        "Authorization": f"Bearer {token}",
        "gid": gid,
        "source":"s3"
    }

    with open(excel_file_path, "rb") as f:
        r = requests.post(
            UPLOAD_URL,
            params={"data_domain_id": data_domain_id},
            headers=headers,
            files={
                "file": (
                    os.path.basename(excel_file_path),
                    f,
                    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
                )
            },
            timeout=300
        )

    if r.status_code >= 400:
        raise RuntimeError(f"Upload error {r.status_code}: {r.text}")

    return r.json()


# =========================
# 4) Export Excel dans /Volumes + upload API : fichier par fichier
# =========================

# crée le dossier du run dans le volume (DBFS path, safe pour dbutils)
run_volume_dir = f"{VOLUME_EXCEL_DIR_DBFS}{run_ts}/"   # dbfs:/Volumes/...
dbutils.fs.mkdirs(run_volume_dir)

upload_success = []
upload_failed = []

for accepted_path in accepted_dest_paths:
    file_name = accepted_path.split("/")[-1]
    base_name = file_name.replace(".parquet", "")
    print(f"\n=== Export+Upload: {accepted_path} ===")

    try:
        # 1) Read + build
        kpi_file = spark.read.parquet(accepted_path)
        kpi2_file = build_kpi2_for_one_file(kpi_file)

        row = kpi2_file.select("USERNAME", "Data Domain", "GID").first()
        if row is None:
            raise ValueError(f"kpi2 is empty for file {accepted_path}")

        gid = str(row["GID"]).lower()
        data_domain_id = str(row["Data Domain"])

        # 2) Export Excel local
        excel_local_for_upload = export_kpi2_to_excel_local_only(kpi2_file.drop("GID"))

        excel_volume_dbfs_path = f"{run_volume_dir}{base_name}.xlsx"
        print(f"⚠️ Copy to Volume skipped (no UC permissions). Intended: {excel_volume_dbfs_path}")

        # 3) Upload
        resp = upload_excel(excel_local_for_upload, data_domain_id=data_domain_id, gid=gid)

        print(f"✅ Uploaded {file_name} -> {resp}")
        print(f"Saved in Volume -> {excel_volume_dbfs_path}")

        upload_success.append({
            "file": accepted_path,
            "gid": gid,
            "data_domain_id": data_domain_id,
            "response": resp
        })

    except Exception as e:
        # Log API/upload/export errors and continue
        err_payload = {
            "run_ts": run_ts,
            "accepted_file": accepted_path,
            "file_name": file_name,
            "error": str(e),
        }
        log_path = f"{s3_logs_api.rstrip('/')}/upload_failed_{run_ts}_{base_name}.json"
        write_text(log_path, json.dumps(err_payload, indent=2, ensure_ascii=False))

        print(f"❌ Upload failed for {file_name} -> {e}")
        print(f"   Log -> {log_path}")

        upload_failed.append({
            "file": accepted_path,
            "error": str(e),
            "log_path": log_path
        })
        continue

print("\n=== Résumé upload ===")
print("Upload OK:", len(upload_success))
print("Upload FAILED:", len(upload_failed))